# 👥 SmartStock – Customer Analytics & Segmentation
### RFM Behavior, Spending Patterns & Repeat Customer Insights
**Objective**: Analyze customer lifetime spend, repeat purchase frequency, Average Customer Value, and RFM behavioral segmentation.


## 1. Load Customer and Order Records


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = os.path.join('..', 'datasets')
df_customers = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
df_sales = pd.read_csv(os.path.join(DATA_DIR, 'sales.csv'))

print(f"Loaded {len(df_customers)} registered customers and {len(df_sales)} sales orders.")
df_customers.head(3)


## 2. Customer Cohort & Repeat Purchase Metrics
Calculate repeat purchase rate and distribution of single vs. multi-time buyers.


In [ ]:
total_customers = len(df_customers)
repeat_customers = len(df_customers[df_customers['order_count'] > 1])
single_order_customers = len(df_customers[df_customers['order_count'] == 1])
zero_order_customers = len(df_customers[df_customers['order_count'] == 0])

repeat_rate = (repeat_customers / total_customers) * 100
avg_spend_per_customer = df_customers['total_spent'].mean()

print("=" * 55)
print(f"👥 Total Registered Customers:    {total_customers}")
print(f"🔄 Repeat Buyers (>1 Order):       {repeat_customers} ({repeat_rate:.1f}%)")
print(f"1️⃣  One-Time Buyers:              {single_order_customers}")
print(f"💤 Inactive / Zero Orders:         {zero_order_customers}")
print(f"💵 Avg Spending per Customer:     ${avg_spend_per_customer:,.2f}")
print("=" * 55)


## 3. Top Spending Customers
Rank top customers by total historical spend and order count.


In [ ]:
top_spenders = df_customers.sort_values(by='total_spent', ascending=False).head(10)

plt.figure(figsize=(12, 6))
bars = plt.barh(top_spenders['name'][::-1], top_spenders['total_spent'][::-1], color='#0284c7')
plt.title('Top 10 Customers by Total Lifetime Spend', fontweight='bold', fontsize=13)
plt.xlabel('Total Spent ($)', fontweight='semibold')
plt.gca().xaxis.set_major_formatter('${x:,.0f}')

for bar in bars:
    w = bar.get_width()
    plt.text(w + 50, bar.get_y() + bar.get_height()/2, f"${w:,.2f}", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

display(top_spenders[['code', 'name', 'total_spent', 'order_count', 'avg_order_value', 'customer_segment']])


## 4. Customer Segmentation (RFM Analysis)
Break down customers into behavioral segments based on frequency, monetary value, and recency.


In [ ]:
segment_stats = df_customers.groupby('customer_segment').agg(
    customer_count=('customer_id', 'count'),
    total_revenue=('total_spent', 'sum'),
    avg_orders=('order_count', 'mean'),
    avg_spend=('total_spent', 'mean')
).reset_index().sort_values(by='total_revenue', ascending=False)

segment_stats['rev_share_pct'] = (segment_stats['total_revenue'] / segment_stats['total_revenue'].sum()) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Donut chart of customer count
colors = ['#3b82f6', '#10b981', '#f59e0b', '#ec4899', '#8b5cf6', '#64748b']
ax1.pie(segment_stats['customer_count'], labels=segment_stats['customer_segment'], autopct='%1.1f%%',
        colors=colors, startangle=140, pctdistance=0.8, wedgeprops=dict(width=0.4, edgecolor='w'))
ax1.set_title('Customer Count by Segment', fontweight='bold', fontsize=12)

# Bar chart of revenue contribution
ax2.bar(segment_stats['customer_segment'], segment_stats['total_revenue'], color=colors[:len(segment_stats)], alpha=0.85)
ax2.set_title('Revenue Contribution by Segment ($)', fontweight='bold', fontsize=12)
ax2.set_ylabel('Total Revenue ($)')
ax2.yaxis.set_major_formatter('${x:,.0f}')
ax2.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

display(segment_stats)


## 5. Spending vs. Order Frequency Scatter Analysis


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_customers['order_count'], df_customers['total_spent'], color='#6366f1', s=80, alpha=0.7, edgecolors='black')
plt.title('Customer Order Frequency vs. Total Spend', fontsize=13, fontweight='bold')
plt.xlabel('Total Orders Placed', fontweight='semibold')
plt.ylabel('Total Spend ($)', fontweight='semibold')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')

# Trendline
m, b = np.polyfit(df_customers['order_count'], df_customers['total_spent'], 1)
x_vals = np.array([df_customers['order_count'].min(), df_customers['order_count'].max()])
plt.plot(x_vals, m*x_vals + b, color='#ef4444', linestyle='--', label=f'Trend (Slope: ${m:.2f}/order)')
plt.legend()
plt.tight_layout()
plt.show()
